# 04 - Anomaly Detection

Detect anomalies in population counts using Z-score, IQR, and rolling window methods. Identify spikes, drops, and missing reporting periods.

**Toolkit modules used**: `cj_data_quality.anomaly`, `cj_data_quality.visualization`

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from cj_data_quality.notebook_utils import setup_notebook
from cj_data_quality.sample_data import generate_and_load
from cj_data_quality.anomaly.time_series_detector import (
    detect_zscore_anomalies,
    detect_iqr_anomalies,
    detect_rolling_anomalies,
    detect_missing_periods,
)
from cj_data_quality.anomaly.spike_detector import detect_spikes, detect_population_anomalies
from cj_data_quality.visualization.plots import plot_anomaly_scatter

setup_notebook()
print("Setup complete.")

## Load and Prepare Data

In [ ]:
df = generate_and_load(n_records=50000, seed=42)

# Aggregate to monthly state-level population for time series analysis
ts = df.dropna(subset=["reporting_date", "total_population"]).copy()
ts["month"] = ts["reporting_date"].dt.to_period("M").dt.to_timestamp()

monthly = ts.groupby(["state_code", "month"]).agg(
    total_population=("total_population", "mean")
).reset_index()

print(f"Monthly aggregation: {monthly.shape}")
monthly.head()

## Z-Score Anomaly Detection

Flag values more than 3 standard deviations from the mean.

In [ ]:
# California time series
ca_monthly = monthly[monthly["state_code"] == "US_CA"].sort_values("month").reset_index(drop=True)

zscore_anomalies = detect_zscore_anomalies(
    ca_monthly, date_col="month", value_col="total_population",
    threshold=2.5, metric_name="total_population", state_code="US_CA"
)

print(f"Z-score anomalies found: {len(zscore_anomalies)}")
for a in zscore_anomalies:
    print(f"  {a.timestamp}: observed={a.observed_value:.0f}, expected={a.expected_value:.0f}, z={a.deviation:.2f}")

In [ ]:
# Visualize
anomaly_dates = [a.timestamp for a in zscore_anomalies]
anomaly_idx = ca_monthly[ca_monthly["month"].dt.date.isin(anomaly_dates)].index.tolist()

fig = plot_anomaly_scatter(
    ca_monthly, "month", "total_population", anomaly_idx,
    title="California Population: Z-Score Anomalies"
)
plt.show()

## IQR-Based Detection

More robust to skewed distributions than Z-scores.

In [ ]:
iqr_anomalies = detect_iqr_anomalies(
    ca_monthly, date_col="month", value_col="total_population",
    multiplier=1.5, metric_name="total_population", state_code="US_CA"
)

print(f"IQR anomalies found: {len(iqr_anomalies)}")
for a in iqr_anomalies:
    print(f"  {a.timestamp}: observed={a.observed_value:.0f}, deviation from fence={a.deviation:.0f}")

## Spike Detection

Detect sudden period-over-period changes (>50% change).

In [ ]:
spikes = detect_spikes(
    ca_monthly, date_col="month", value_col="total_population",
    pct_change_threshold=0.5, metric_name="total_population", state_code="US_CA"
)

print(f"Spikes/drops detected: {len(spikes)}")
for s in spikes:
    direction = "SPIKE" if s.anomaly_type.value == "sudden_spike" else "DROP"
    print(f"  {s.timestamp}: {direction} — observed={s.observed_value:.0f}, previous={s.expected_value:.0f}, change={s.deviation:.1%}")

## Cross-State Population Anomalies

In [ ]:
pop_anomalies = detect_population_anomalies(
    monthly, date_col="month", population_col="total_population",
    state_col="state_code", threshold=2.5
)

print(f"Cross-state population anomalies: {len(pop_anomalies)}")

anomaly_by_state = {}
for a in pop_anomalies:
    anomaly_by_state.setdefault(a.state_code, []).append(a)

for state, anomalies in sorted(anomaly_by_state.items()):
    print(f"  {state}: {len(anomalies)} anomalies")

## Missing Period Detection

In [ ]:
# Check a few states for missing monthly reports
for state in ["US_CA", "US_AK", "US_WY"]:
    state_ts = monthly[monthly["state_code"] == state].sort_values("month")
    if len(state_ts) > 0:
        missing = detect_missing_periods(
            state_ts, date_col="month", expected_freq="MS",
            metric_name="monthly_report", state_code=state
        )
        print(f"{state}: {len(missing)} missing months out of expected cadence")

## Key Findings

1. **Z-score detection** successfully catches the injected population spikes
2. **IQR method** provides a more robust alternative for skewed distributions
3. **Spike detection** identifies sudden period-over-period changes that may indicate data pipeline issues
4. **Missing period detection** flags gaps in expected reporting cadence
5. **Multiple methods** provide defense-in-depth — different anomaly types may be caught by different detectors

For Recidiviz, this would serve as an automated early warning system when state data feeds deviate from expected patterns.